# 04. Memoization

**Tier:** Deep. **Why you'd use this:** A recursive fib-style function recomputes the same smaller answers many times. A cache stores each answer once so later calls reuse it.

**After this lesson you can:** recognize a cache hit; wire `cache` so the recursive calls go through it.
About 25 minutes. **Uses:** [Base case](01-base-case-lesson.ipynb), [The limit](03-limit-lesson.ipynb), [itertools and cache](../08-collections-module/05-itertools-lesson.ipynb)


## Quick review

1. Does Python reuse the current stack frame for a tail call?
2. What is one way to walk a tree without deep recursion?

<details><summary>Answers</summary>

No — each call still pushes a frame. Keep an explicit stack (a list) in a loop.
</details>


## The idea

Some recursive calls repeat themselves. `choose(n, k)`, the number of ways to pick k items out of n, is `choose(n - 1, k - 1) + choose(n - 1, k)`, and those two branches ask for the same smaller pairs again. The answers do not change. Computing them twice is the whole cost.

Memoization means storing a result the first time you compute it, keyed by the arguments. `functools.cache` wraps a function in that store. A later call with the same arguments is a hit: the wrapper returns the stored value and does not enter the function. The first call with those arguments is a miss.

The wrapper only helps if the recursive calls use the wrapper. The name inside the function has to be the wrapped function, not the original.

<details><summary>Explain more</summary>

`cache` returns a new function. Assign it back to the same name the body calls: `choose = cache(choose)`. The `@` spelling is the same assignment, and it arrives with decorators. Until then, call `cache` yourself. The arguments must be hashable, because the store is a dict and a dict key must be hashable.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is a hand-rolled memo table, provided as a wrapper. A `Map` from the argument pair to the result is the same design. The wrapper does the lookup so the recurrence stays readable.
</details>


## See it work

`choose(n, k)` is 1 when `k` is 0 or `k` is `n`: there is one way to pick nothing, and one way to pick everything. Otherwise it is the sum of "pick this item" and "skip this item". The pair `(n, k)` shows up on many branches. Count the calls, then store the answers.


In [1]:
calls = 0


def choose(n: int, k: int) -> int:
    global calls
    calls += 1
    if k == 0 or k == n:
        return 1
    return choose(n - 1, k - 1) + choose(n - 1, k)


print(choose(6, 3), calls)


20 39


In [2]:
import functools

calls = 0


def choose(n: int, k: int) -> int:
    global calls
    calls += 1
    if k == 0 or k == n:
        return 1
    return choose(n - 1, k - 1) + choose(n - 1, k)


choose = functools.cache(choose)
print(choose(6, 3), calls)
print(choose.cache_info())


20 15
CacheInfo(hits=4, misses=15, maxsize=None, currsize=15)


Both cells print `20`, which is the right number of ways. The first cell needs 39 calls. The second needs 15, and `cache_info` reports 15 misses and 4 hits. Fifteen is the number of distinct pairs the recurrence actually touches. Four calls found a pair that was already stored and did not enter the function, which is why the counter did not move for them.

| pair asked more than once | why it repeats |
| --- | --- |
| smaller `(a, b)` under two parents | the "pick" branch and the "skip" branch overlap |
| `(n, 0)` and `(n, n)` | base cases, stored after the first time |

```
choose(4, 2)
├── choose(3, 1)
│   ├── choose(2, 0)
│   └── choose(2, 1)
└── choose(3, 2)
    ├── choose(2, 1)   same pair again
    └── choose(2, 2)
```


## Check yourself

Answer these from the example above before you run the cell.


1. Why is the wrong wiring's result still 20?
2. Why does `cache_info().hits` stay 0 on a function that never repeats an argument?
3. What does `choose.cache_clear()` change about the next call's counter?

<details><summary>Answers</summary>

1. The raw function implements the recurrence correctly. The wrapper was only sitting on the outside call. Correctness never depended on the dict. The dict only skips work.
2. A hit requires a second call with the same key. If every key is new, every call is a miss, and hits stays 0 while the dict grows.
3. `cache_clear` drops the dict. The next call is a miss again, the body runs, and the counter moves as it did on a cold start.
</details>


In [3]:
import functools


def double(n: int) -> int:
    print("miss", n)
    return n * 2


remembered = functools.cache(double)
print("first", remembered(3))
print("second", remembered(3))
print("hits", remembered.cache_info().hits)


miss 3
first 6
second 6
hits 1


<details><summary>Check your answers</summary>

- `miss 3` prints once. The second call does not enter `double`.
- `first 6` and `second 6` are the same stored return value.
- `hits` is 1. The first call was a miss, and misses are not hits. `cache_info` counts them separately.
</details>


## Why it works this way

`cache` builds a dict whose key is the tuple of arguments and whose value is the return value. On a hit the wrapper returns that value and the body does not run, so a counter inside the body stays still. On a miss the wrapper calls the body, stores whatever it returns, and the body may itself call the wrapper. Those inner calls consult the same dict. That is the whole speedup: the call tree becomes a table of pairs, each filled once.

Without the dict, `choose(n - 1, k)` and `choose(n - 1, k - 1)` both recompute the pairs under them. The number of calls grows exponentially in n. With the dict, each pair `(a, b)` with `a <= n` is computed once. There are on the order of `n * k` such pairs in the rectangle the original call can reach, so time and extra memory are both proportional to that rectangle, not to the tree. The dict lives as long as the wrapped function does. A function that is called with a huge spread of arguments keeps every result.

The name lookup is the part people skip. The body of `choose` looks up the global name `choose` when it calls itself. If that name is the wrapper, the lookup hits the dict. If you wrote `remembered = cache(choose)` and left the global `choose` pointing at the raw function, the outer call is wrapped and every inner call is not. The result is still 20. The counter is still 39. The bug is silent because the function is correct and only the cost is wrong. The common mistake section prints that pair.

Arguments have to be hashable, since they become a dict key. A list, a dict, or a set cannot be a key. An int can. A tuple of ints can. That is the same rule as any dict you write by hand, and it is why `cache` around a function that takes a list raises `TypeError` on the first call.

The design reason is to keep the recurrence you can check by hand, and move the repeated work into a table. A loop that fills a table from small pairs up to `(n, k)` is the same table without a call stack. Use that bottom-up loop when the recursion would be deep enough to hit the frame cap, or when you want to drop rows you no longer need. `cache` is the right tool when the recursive spelling is the one you trust and the depth stays under the cap. The frame cap is unchanged by the wrapper: a deep chain of unique arguments still builds a frame per step. Memoization removes repeated calls. It does not remove the stack.


## When to use it

Use `cache` when the same arguments come back and the result depends only on those arguments. `choose` is the pattern. A tree walk that visits each node once is not: every call is a miss, and the dict holds a copy of the tree for no hits. A function that reads a global you change between calls is not either. The wrapper will hand back the result from the first global, because the global is not part of the key.

Do not wrap a function whose argument is a list. Convert the list to a tuple first, or memoize on a count or an index instead of on the list. Do not use `cache` to survive a chain of unique calls a thousand deep. That is a stack problem, and the previous lesson's list is the fix. A bottom-up table is the alternative when you want the memo and you also want a loop, so the depth stays constant.

Call `cache_clear` if you need to drop the stored answers, for a test or because the function's other inputs changed. Clearing is part of the wrapper, not of the original function.


## Common mistake

Binding the wrapper to a new name leaves the recursive calls on the raw function. `choose(6, 3)` is still 20. The call counter shows the full tree. You saved nothing, and no test that checks only the return value will notice.


In [4]:
import functools

calls = 0


def choose(n: int, k: int) -> int:
    global calls
    calls += 1
    if k == 0 or k == n:
        return 1
    return choose(n - 1, k - 1) + choose(n - 1, k)


remembered = functools.cache(choose)
print(remembered(6, 3), calls)


20 39


That prints `20 39`. The fix assigns the wrapper back to `choose`, so the name the body calls is the name that holds the dict. The answer stays 20. The counter drops, because repeated pairs never re-enter the body.


In [5]:
import functools

calls = 0


def choose(n: int, k: int) -> int:
    global calls
    calls += 1
    if k == 0 or k == n:
        return 1
    return choose(n - 1, k - 1) + choose(n - 1, k)


choose = functools.cache(choose)
print(choose(6, 3), calls)


20 15


A list argument fails for a different reason. The last line of the traceback is `TypeError`, and it says the list is unhashable. The wrapper tried to use the list as a dict key. Read that last line first.


In [6]:
import functools


def total(nums: list[int]) -> int:
    return sum(nums)


total = functools.cache(total)
total([1, 2, 3])


TypeError: unhashable type: 'list'

## Try it

After a cached `choose(6, 3)`, call `choose.cache_clear()`, then call `choose(6, 3)` again and print the counter and `cache_info()`. Why does the second call add more than zero to the counter, and which field of `cache_info` tells you the dict was empty?


## Recap

- `cache` stores return values in a dict keyed by the arguments.
- The recursive name has to point at the wrapper, or inner calls skip the dict.
- Repeated arguments get cheaper. A deep chain of new arguments still uses a frame per step.

Next: [practice](04-memoization-practice.ipynb).
